# Kustomize 02: patches

Two patch dialects: strategic merge (a partial object merged by the Kubernetes rules) and JSON 6902 (operations on paths). Both target resources by kind, name and labels.


In [ ]:
cd /source/work/kustomize-lab/overlays/prod
cat > resources.yaml <<'YAML'
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
spec:
  template:
    spec:
      containers:
        - name: web
          resources:
            requests: {cpu: 100m, memory: 64Mi}
            limits: {memory: 128Mi}
YAML
kustomize edit add patch --path resources.yaml && kustomize build . | yq 'select(.kind == "Deployment") | .spec.template.spec.containers[0].resources'


In [ ]:
cd /source/work/kustomize-lab/overlays/prod
cat > probe.json <<'JSON'
[
  {"op": "add", "path": "/spec/template/spec/containers/0/readinessProbe", "value": {"httpGet": {"path": "/health", "port": "http"}}},
  {"op": "replace", "path": "/spec/replicas", "value": 4}
]
JSON
kustomize edit add patch --path probe.json --kind Deployment --name web && kustomize build . | yq 'select(.kind == "Deployment") | .spec.replicas, .spec.template.spec.containers[0].readinessProbe'


The companion repository uses inline JSON patches with `target:` selectors so an overlay stays one file. Compare dev and prod there:


In [ ]:
export HOME=/tmp
mkdir -p /source/work && cd /source/work && [ -d gitops-renderers ] || git clone -q --recurse-submodules https://github.com/cznewt/gitops-renderers.git
cd /source/work/gitops-renderers/examples/01-kustomize && diff overlays/dev/kustomization.yaml overlays/prod/kustomization.yaml || true


In [ ]:
cd /source/work/gitops-renderers/examples/01-kustomize
diff <(kustomize build --enable-helm overlays/dev | yq 'select(.kind == "Deployment" and .metadata.name == "web") | .spec') <(kustomize build --enable-helm overlays/prod | yq 'select(.kind == "Deployment" and .metadata.name == "web") | .spec') || true
